In [ ]:
#@title 1. Connect to Drive & Define File Paths
from google.colab import drive
import os

# Prompt to authorize Colab to access your Google Drive
drive.mount('/content/drive')
#os.chdir('/content/drive/MyDrive/NPL2025_Proj')

# Define the path to data folder in Google Drive
save_dir = '/content/drive/MyDrive/NPL2025_Proj'

# Define the full paths for your input and output files
INPUT_TRAIN_FILE = os.path.join(save_dir, "train_redacted.jsonl")
INPUT_DEV_FILE = os.path.join(save_dir, "dev_redacted.jsonl")

OUTPUT_TRAIN_FILE = os.path.join(save_dir, "train_rehydrated.jsonl")
OUTPUT_DEV_FILE = os.path.join(save_dir, "dev_rehydrated.jsonl")

# Verify that the input files exist
if os.path.exists(INPUT_TRAIN_FILE) and os.path.exists(INPUT_DEV_FILE):
    print("Successfully located redacted files in Google Drive.")
    print(f"Input train file: {INPUT_TRAIN_FILE}")
    print(f"Input dev file:   {INPUT_DEV_FILE}")
else:
    print("Error: Could not find 'train_redacted.jsonl' or 'dev_redacted.jsonl' at the specified path.")


In [ ]:
#@title 2. Install Required Libraries
# This cell installs the Python packages needed for the rehydration script.
!pip install -q requests==2.32.5 beautifulsoup4==4.12.3 Markdown==3.6 tqdm==4.66.4
print("Libraries installed.")

In [ ]:
#@title 3. Rehydrate Data from Drive and Save to Drive
import requests
from tqdm.notebook import tqdm
from bs4 import BeautifulSoup
from markdown import markdown
import json
import re

# --- Preprocessing functions ---
def markdown_to_text(markdown_string):
    html = markdown(markdown_string)
    html = re.sub(r'<pre>(.*?)</pre>', ' ', html)
    html = re.sub(r'<code>(.*?)</code >', ' ', html)
    soup = BeautifulSoup(html, "html.parser")
    text = ' '.join(soup.findAll(string=True))
    return text

def replace_urls(x, url_replacement_token='<URL>'):
    return re.sub("http(.+)?(\W|$)", url_replacement_token, x)

def replace_ss_prefix(x):
    return re.sub(r'^\W*(summary statement|submission statement|ss)[^a-zA-Z]*',"",  x, flags=re.I|re.U).strip()

def preprocess(x):
    return replace_ss_prefix(replace_urls(markdown_to_text(x)))

# --- Main rehydration logic ---
def rehydrate_comments(input_file, output_file_path):
    ids_to_fetch_with_prefix = []
    original_data_map = {}
    with open(input_file, 'r') as f:
        for line in f:
            item = json.loads(line)
            if '_id' in item and item['_id'].startswith('t1_'):
                comment_id_with_prefix = item['_id']
                ids_to_fetch_with_prefix.append(comment_id_with_prefix)
                original_data_map[comment_id_with_prefix] = item

    base_url = "https://arctic-shift.photon-reddit.com/api/comments/ids"
    fields = "body,subreddit,id"
    rehydrated_data = []

    for i in tqdm(range(0, len(ids_to_fetch_with_prefix), 500), desc=f"Rehydrating {os.path.basename(input_file)}"):
        batch_ids_with_prefix = ids_to_fetch_with_prefix[i:i + 500]
        batch_ids_without_prefix = [cid[3:] for cid in batch_ids_with_prefix]
        params = {"ids": ",".join(batch_ids_without_prefix), "fields": fields}

        try:
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            api_response = response.json()

            if "data" in api_response:
                rehydrated_comments_batch = api_response["data"]
                rehydrated_map = {c['id']: c for c in rehydrated_comments_batch if c.get('body', '[deleted]').strip() not in ['[deleted]', '[removed]']}

                for cid_with_prefix in batch_ids_with_prefix:
                    cid_without_prefix = cid_with_prefix[3:]
                    if cid_without_prefix in rehydrated_map:
                        rehydrated_comment = rehydrated_map[cid_without_prefix]
                        original_item = original_data_map[cid_with_prefix]
                        merged_item = {
                            "_id": f"t1_{rehydrated_comment['id']}",
                            "text": preprocess(rehydrated_comment['body']),
                            "subreddit": rehydrated_comment['subreddit'],
                            "conspiracy": original_item.get("conspiracy"),
                            "markers": original_item.get("markers"),
                            "annotator": original_item.get("annotator")
                        }
                        rehydrated_data.append(merged_item)
        except Exception as e:
            print(f"An error occurred: {e}")
            continue

    with open(output_file_path, 'w') as outfile:
        for item in rehydrated_data:
            outfile.write(json.dumps(item) + '\n')
    print(f" Rehydrated data saved to your Google Drive at: '{output_file_path}'")

# --- Run rehydration for both files using the paths from Cell 1 ---
rehydrate_comments(INPUT_TRAIN_FILE, OUTPUT_TRAIN_FILE)
rehydrate_comments(INPUT_DEV_FILE, OUTPUT_DEV_FILE)

print("\n Process complete!!!!!!!!")
